In [ ]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas scikit-learn matplotlib seaborn -q

import platform
import warnings
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# 재현성: 같은 난수를 항상 같게 만듭니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록)
system = platform.system()
if system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)


준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.4.6
pandas: 3.0.3


In [2]:
# ─────────────────────────────────────────────
# 모두마켓 이번 달 주문 데이터 생성 — 자기완결적 스냅샷
# (지난 노드에서 다룬 오염 요소를 적당히 섞어 둡니다)
# ─────────────────────────────────────────────
np.random.seed(42)
n = 1500

regions = np.random.choice(["서울", "경기", "부산", "인천", "대구"], n, p=[0.4, 0.25, 0.15, 0.1, 0.1])
membership = np.random.choice(["basic", "silver", "gold", "vip"], n, p=[0.5, 0.25, 0.15, 0.1])
channels = np.random.choice(["web", "app", "app ", "APP"], n, p=[0.5, 0.4, 0.05, 0.05])
categories = np.random.choice(["패션", "뷰티", "식품", "가전", "도서"], n)

prices = np.random.choice([9900, 19900, 29900, 49900, 89900, 129900, 249900], n,
                          p=[0.2, 0.25, 0.2, 0.15, 0.1, 0.06, 0.04])
quantities = np.random.choice([1, 1, 1, 2, 2, 3], n)
amount = (prices * quantities).astype(float)

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n + 1)],
    "customer_age": np.random.normal(35, 9, n).round().astype(int),
    "region": regions,
    "membership": membership,
    "channel": channels,
    "category": categories,
    "price": prices.astype(float),
    "quantity": quantities,
    "amount": amount,
})

# 오염 심기: 결측·이상치·표기 혼재
orders.loc[np.random.choice(n, 60, replace=False), "amount"] = np.nan
orders.loc[np.random.choice(n, 30, replace=False), "customer_age"] = np.nan
orders.loc[5, "customer_age"] = 999             # 입력 실수성 이상치
orders.loc[8, "customer_age"] = -3              # 불가능한 음수
orders.loc[12, "quantity"] = 80                 # 비정상적으로 큰 주문
orders.loc[20, "region"] = " 서울 "             # 앞뒤 공백
orders.loc[21, "region"] = "Seoul"              # 영문 표기
orders.loc[40, "membership"] = "VIP"            # 대소문자 혼재

print("이번 달 주문 데이터 준비 완료:", orders.shape)
orders.head()

이번 달 주문 데이터 준비 완료: (1500, 9)


,order_id,customer_age,region,membership,channel,category,price,quantity,amount
0,O00001,30.0,서울,silver,app,패션,19900.0,1,NaN
1,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0
2,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0
3,O00004,24.0,경기,basic,app,뷰티,19900.0,1,NaN
4,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0


In [3]:
# 예제: '단계 변수'로 쓴 정제와 'chaining'으로 쓴 정제 — 결과는 같음

# (a) 단계 변수 방식 (전통)
step1 = orders.dropna(subset=["amount", "customer_age"])
step2 = step1[(step1["customer_age"] > 0) & (step1["customer_age"] < 120)]
step3 = step2.assign(
    region_clean=step2["region"].str.strip().replace({"Seoul": "서울"}),
    amount_log=np.log1p(step2["amount"]),
)
clean_a = step3.sort_values("amount", ascending=False).reset_index(drop=True)

# (b) method chaining 방식 (체이닝)
clean_b = (
    orders
    .dropna(subset=["amount", "customer_age"])
    .query("0 < customer_age < 120")
    .assign(
        region_clean=lambda d: d["region"].str.strip().replace({"Seoul": "서울"}),
        amount_log=lambda d: np.log1p(d["amount"]),
    )
    .sort_values("amount", ascending=False)
    .reset_index(drop=True)
)

# 두 결과가 같은가?
print("두 방식 결과 동일?:", clean_a.equals(clean_b))
print("정제 후 행 수:", clean_b.shape[0])
clean_b.head(3)

두 방식 결과 동일?: True
정제 후 행 수: 1411


,order_id,customer_age,region,membership,channel,category,price,quantity,amount,region_clean,amount_log
0,O00659,33.0,경기,basic,app,도서,249900.0,3,749700.0,경기,13.52743
1,O00096,39.0,경기,silver,app,뷰티,249900.0,3,749700.0,경기,13.52743
2,O00531,49.0,인천,basic,web,가전,249900.0,3,749700.0,인천,13.52743


In [4]:
# 예제: query로 같은 필터 작성
# 1) 전통 방식
cond = (orders["customer_age"] > 0) & (orders["customer_age"] < 120) & (orders["amount"] >= 30000)
trad = orders[cond]

# 2) query 방식
q = orders.query("0 < customer_age < 120 and amount >= 30000")

print("동일?", trad.equals(q))
print("건수:", len(q))

동일? True
건수: 776


In [4]:
# 스스로 해보자! (4)
# 아래 빈칸(___)을 채우고 실행해보세요.

result = (
     orders
     .dropna(subset=["amount"])
     .query("0 < customer_age < 120")
     .assign(
         channel_clean=lambda d: d["channel"].str.strip().str.lower(),
         amount_log=lambda d: np.log1p(d["amount"]),
     )
     .sort_values("amount", ascending=False)
     .reset_index(drop=True)
 )
print(result.shape)
result.head(3)

(1411, 11)


,order_id,customer_age,region,membership,channel,category,price,quantity,amount,channel_clean,amount_log
0,O00659,33.0,경기,basic,app,도서,249900.0,3,749700.0,app,13.52743
1,O00096,39.0,경기,silver,app,뷰티,249900.0,3,749700.0,app,13.52743
2,O00531,49.0,인천,basic,web,가전,249900.0,3,749700.0,web,13.52743


In [5]:
# 예제: 단계마다 함수로 빼기 — '한 함수 = 한 책임'
def clean_strings(df):
    # 공백·대소문자·표기 혼재를 정리합니다.
    return df.assign(
        region=df["region"].str.strip().replace({"Seoul": "서울"}),
        membership=df["membership"].str.lower(),
        channel=df["channel"].str.strip().str.lower(),
    )

def drop_invalid(df, age_min=0, age_max=120, qty_max=20):
    # 불가능한 값(나이 범위·과대 수량)과 결측을 제거합니다.
    return (
        df
        .dropna(subset=["amount", "customer_age"])
        .query("@age_min < customer_age < @age_max")
        .query("quantity <= @qty_max")
    )

def add_features(df):
    # 분석에 쓸 파생 컬럼을 추가합니다.
    return df.assign(
        amount_log=lambda d: np.log1p(d["amount"]),
        is_premium=lambda d: d["membership"].isin(["gold", "vip"]).astype(int),
    )

print("세 함수가 준비됐습니다. 다음 셀에서 한 줄로 조립합니다.")

세 함수가 준비됐습니다. 다음 셀에서 한 줄로 조립합니다.


In [6]:
# 예제: 한 줄로 조립
cleaned = (
    orders
    .pipe(clean_strings)
    .pipe(drop_invalid, age_min=10, age_max=80, qty_max=10)
    .pipe(add_features)
)

print("원본:", orders.shape, "→ 정제 후:", cleaned.shape)
cleaned.head(3)

원본: (1500, 9) → 정제 후: (1404, 11)


,order_id,customer_age,region,membership,channel,category,price,quantity,amount,amount_log,is_premium
1,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0,13.527430,0
2,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0,9.898525,0
4,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0,9.898525,0


In [7]:
# 예제: 한 줄로 조립
cleaned = (
    orders
    .pipe(clean_strings)
    .pipe(drop_invalid, age_min=10, age_max=80, qty_max=10)
    .pipe(add_features)
)

print("원본:", orders.shape, "→ 정제 후:", cleaned.shape)
cleaned.head(3)

원본: (1500, 9) → 정제 후: (1404, 11)


,order_id,customer_age,region,membership,channel,category,price,quantity,amount,amount_log,is_premium
1,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0,13.527430,0
2,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0,9.898525,0
4,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0,9.898525,0


In [8]:
# 예제: 단계마다 함수로 빼기 — '한 함수 = 한 책임'
def clean_strings(df):
    # 공백·대소문자·표기 혼재를 정리합니다.
    return df.assign(
        region=df["region"].str.strip().replace({"Seoul": "서울"}),
        membership=df["membership"].str.lower(),
        channel=df["channel"].str.strip().str.lower(),
    )

def drop_invalid(df, age_min=0, age_max=120, qty_max=20):
    # 불가능한 값(나이 범위·과대 수량)과 결측을 제거합니다.
    return (
        df
        .dropna(subset=["amount", "customer_age"])
        .query("@age_min < customer_age < @age_max")
        .query("quantity <= @qty_max")
    )

def add_features(df):
    # 분석에 쓸 파생 컬럼을 추가합니다.
    return df.assign(
        amount_log=lambda d: np.log1p(d["amount"]),
        is_premium=lambda d: d["membership"].isin(["gold", "vip"]).astype(int),
    )

print("세 함수가 준비됐습니다. 다음 셀에서 한 줄로 조립합니다.")

세 함수가 준비됐습니다. 다음 셀에서 한 줄로 조립합니다.


In [9]:
# 예제: 한 줄로 조립
cleaned = (
    orders
    .pipe(clean_strings)
    .pipe(drop_invalid, age_min=10, age_max=80, qty_max=10)
    .pipe(add_features)
)

print("원본:", orders.shape, "→ 정제 후:", cleaned.shape)
cleaned.head(3)

원본: (1500, 9) → 정제 후: (1404, 11)


,order_id,customer_age,region,membership,channel,category,price,quantity,amount,amount_log,is_premium
1,O00002,24.0,대구,basic,app,가전,249900.0,3,749700.0,13.527430,0
2,O00003,45.0,부산,basic,web,패션,19900.0,1,19900.0,9.898525,0
4,O00005,41.0,서울,basic,app,패션,19900.0,1,19900.0,9.898525,0


In [10]:
# 예제: 인코딩/스케일링 함수 추가
from sklearn.preprocessing import RobustScaler

def encode_categories(df):
    # membership은 Ordinal, region·channel·category는 One-Hot.
    order_map = {"basic": 1, "silver": 2, "gold": 3, "vip": 4}
    out = df.assign(membership_ord=df["membership"].map(order_map))
    # One-Hot
    out = pd.concat(
        [out,
         pd.get_dummies(out["region"], prefix="region", dtype=int),
         pd.get_dummies(out["channel"], prefix="ch", dtype=int),
         pd.get_dummies(out["category"], prefix="cat", dtype=int)],
        axis=1
    )
    return out

def scale_numeric(df, cols=("customer_age", "amount", "quantity")):
    # 수치형 컬럼을 RobustScaler로 스케일링.
    scaler = RobustScaler()
    scaled = scaler.fit_transform(df[list(cols)])
    scaled_df = pd.DataFrame(scaled, columns=[f"{c}_scaled" for c in cols], index=df.index)
    return pd.concat([df, scaled_df], axis=1)

# 전체 파이프라인 한 줄
pipeline_result = (
    orders
    .pipe(clean_strings)
    .pipe(drop_invalid, age_min=10, age_max=80, qty_max=10)
    .pipe(add_features)
    .pipe(encode_categories)
    .pipe(scale_numeric)
)

print("최종 shape:", pipeline_result.shape)
print("새로 생긴 컬럼 일부:", [c for c in pipeline_result.columns if "scaled" in c or c.startswith("region_")][:8])

최종 shape: (1404, 27)
새로 생긴 컬럼 일부: ['region_경기', 'region_대구', 'region_부산', 'region_서울', 'region_인천', 'customer_age_scaled', 'amount_scaled', 'quantity_scaled']


In [11]:
# 스스로 해보자! (5)
def add_amount_class(df):
     return df.assign(
         amount_class=np.where(df["amount"] >= 100_000, "high", "low")
     )

orders_v2 = (
     orders
     .pipe(clean_strings)
     .pipe(drop_invalid)
     .pipe(add_features)
     .pipe(add_amount_class)   # 새 함수 끼우기
 )
print(orders_v2[["amount", "amount_class"]].head())
print(orders_v2["amount_class"].value_counts())

     amount amount_class
1  749700.0         high
2   19900.0          low
4   19900.0          low
6   59700.0          low
7  389700.0         high
amount_class
low     1162
high     248
Name: count, dtype: int64
